#### Environment Setup
Creates the `<catalog>` catalog and `<catalog>.forwardstep` schema, the `landing` volume, and its
seven subfolders.

Parameterized via a `catalog` widget — this is one-time-per-environment setup, run once against
`dev`, once against `uat`, once against `prod`. Every statement below is idempotent
(`IF NOT EXISTS` / `mkdirs`), so a re-run against an environment that already exists is a safe
no-op, not an error.

In [0]:
dbutils.widgets.text("catalog", "dev", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Running environment setup for catalog: {catalog}")

Running environment setup for catalog: dev


Fixed from the original: every reference below used to be hardcoded to `dev` — this notebook
predates the project's catalog-parameterization convention (`get_catalog()`, locked in
Module 6) and never got updated. Running it unmodified against `uat` or `prod` would have
silently kept touching `dev` instead. `%sql` magic cells can't take a Python f-string
substitution directly, so these are `spark.sql(f"...")` calls instead — same pattern already
used everywhere else parameterization matters in this project.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

DataFrame[]

In [0]:
display(spark.sql(f"SHOW CATALOGS LIKE '{catalog}'"))

catalog
dev


In [0]:
# Decision 1 (unchanged from original): a dedicated schema for this project, isolated from dev.dbx_course
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.forwardstep")

DataFrame[]

In [0]:
display(spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE 'forwardstep'"))

databaseName
forwardstep


In [0]:
# Decision 2 (unchanged from original): one landing volume for all five sources, not five separate volumes
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.forwardstep.landing")

DataFrame[]

#### Create the seven landing subfolders
Three CDC subfolders (orders, order_items, customers) and four file-based subfolders
(products, categories, clickstream, inventory) — one folder per source, inside the single
`landing` volume.

In [0]:
landing_root = f"/Volumes/{catalog}/forwardstep/landing"

subfolders = [
    "orders_cdc",
    "order_items_cdc",
    "customers_cdc",
    "products",
    "categories",
    "clickstream",
    "inventory",
]

for folder in subfolders:
    dbutils.fs.mkdirs(f"{landing_root}/{folder}")
    print(f"Created: {landing_root}/{folder}")

Created: /Volumes/dev/forwardstep/landing/orders_cdc
Created: /Volumes/dev/forwardstep/landing/order_items_cdc
Created: /Volumes/dev/forwardstep/landing/customers_cdc
Created: /Volumes/dev/forwardstep/landing/products
Created: /Volumes/dev/forwardstep/landing/categories
Created: /Volumes/dev/forwardstep/landing/clickstream
Created: /Volumes/dev/forwardstep/landing/inventory


#### Verify the landing zone

In [0]:
display(dbutils.fs.ls(landing_root))

path,name,size,modificationTime
dbfs:/Volumes/dev/forwardstep/landing/categories/,categories/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/clickstream/,clickstream/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/customers_cdc/,customers_cdc/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/inventory/,inventory/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/order_items_cdc/,order_items_cdc/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/orders_cdc/,orders_cdc/,0,1792956542952
dbfs:/Volumes/dev/forwardstep/landing/products/,products/,0,1792956542953


All seven folders should be listed above. On a first run against a new catalog they'll be
empty; on a re-run against an environment that already has data, this step is a safe no-op.

####Scaffold the GitHub
Setup complete project structure now, using the terminal built into this Git folder.
```bash
mkdir -p setup
mkdir -p pipelines/ingestion_pipeline/transformations
mkdir -p pipelines/ingestion_pipeline/utilities
mkdir -p pipelines/ingestion_pipeline/tests
mkdir -p pipelines/transformation_pipeline/transformations
mkdir -p pipelines/transformation_pipeline/utilities
mkdir -p pipelines/transformation_pipeline/tests
mkdir -p jobs
mkdir -p bundle/resources
mkdir -p .github/workflows
```

####Placeholder files
Git does not track empty folders. If we commit right now, most of these empty folders will simply disappear from GitHub. So, we place one placeholder file inside each empty folder.

```bash
find setup pipelines jobs bundle .github -type d -empty -exec touch {}/.gitkeep \;
```

####Let us also create three files we already know we will need.
```bash
touch README.md
touch requirements.txt
touch data_generator.py
```

####Commit to GitHub
Make sure you have repo and workflow access to your GitHub PAT
